[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/courses/binomial_theorem_course.ipynb)

# Binomial theorem full course: practice notebook

One practice question for every teaching part of the video, in the same order, and two cells you can move yourself:
Pascal's triangle, growing a row at a time, and the general term of any bracket, stepping through its terms. Work
each question out on paper first, then type your answer into the box and press **check**. It tells you whether
you're right. Press **show working** if you'd like to see the steps.

**How to use this notebook.** The grey boxes are cells. Click the first one and press **Shift + Enter** (or the ▶ Run
button) to run it. Run each cell in turn, from the top down. When sliders appear, move them and watch: the cell runs
again by itself. Change any number and run the cell again: nothing can break. It works in your browser (no install)
and in Google Colab.

**Typing answers.** Powers with `^`: `x^3 + 12x^2 + 48x + 64` (the terms in any order). Whole numbers without commas
or spaces: `96059601`. Decimals as decimals: `1.267`. You can type the working too: `6*5*4`, `5!`, `10C3` for 10
choose 3. For several answers, separate them with commas, in the order the question asks. You can also put your
answer between the quotes in a cell's last line, `practice(1, "")`, and run the cell: it checks straight away.

**The cheat sheet.** Every result from the course on two pages:
[binomial theorem cheat sheet (PDF)](https://nammaschool.github.io/interactive-learning/cheatsheets/binomial_theorem_cheat_sheet.pdf).

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look and the checker: run this cell first.
import math, re
from math import comb, factorial
from itertools import product
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from sympy.parsing.sympy_parser import parse_expr, standard_transformations, implicit_multiplication_application, convert_xor
from IPython.display import display
from ipywidgets import interact, IntSlider, Text, Button, Output, HBox, VBox, Layout

SLATE, SLATE2 = "#1f2b28", "#2a3a36"        # the slate ground
CHALK, DIM, GHOST = "#ece6d6", "#93a097", "#3b4c47"   # chalk, chalk smudge, ghost lines
CORAL = "#e3896b"                           # the one key thing in each figure
SKY, MINT, GOLD = "#6fb6bb", "#a0c27c", "#e8c56b"     # a · b · the answer

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": False, "font.size": 13, "axes.titlesize": 15, "legend.fontsize": 12,
    "lines.linewidth": 2.4, "figure.dpi": 100, "mathtext.fontset": "dejavusans",
})

RULES = standard_transformations + (implicit_multiplication_application, convert_xor)
x = sp.Symbol("x")
NAMES = {"x": x, "binomial": sp.binomial, "ff": sp.ff, "factorial": sp.factorial, "sqrt": sp.sqrt}
SUPS, SUBS = "⁰¹²³⁴⁵⁶⁷⁸⁹", "₀₁₂₃₄₅₆₇₈₉"
def sup(n): return "".join(SUPS[int(d)] for d in str(n))

def prep(s):
    """Typed symbols → Python: − × ÷ ² ³ ⁴, 10C3 (choose), 650 000 → 650000."""
    s = s.replace("−", "-").replace("×", "*").replace("÷", "/").replace("·", "*")
    s = re.sub(f"([{SUPS}]+)\\s*([Cc])\\s*([{SUBS}]+)",
               lambda m: "".join(str(SUPS.index(c)) for c in m[1]) + "C" + "".join(str(SUBS.index(c)) for c in m[3]), s)
    s = re.sub(r"(\d+)\s*[Cc]\s*(\d+)", r"binomial(\1,\2)", s)
    s = re.sub(f"[{SUPS}]+", lambda m: "^" + "".join(str(SUPS.index(c)) for c in m[0]), s)
    return re.sub(r"(?<=\d) (?=\d{3}\b)", "", s)

def read_one(p): return parse_expr(p, local_dict=NAMES, transformations=RULES)

def read(s):
    """Your typed answer (a list, one for each answer)."""
    return [read_one(prep(p)) for p in s.split(",") if p.strip()]

def same(got, want, tol=1e-9):
    try:
        d = sp.expand(sp.sympify(got) - want)
        if d.free_symbols: return False                                 # a different expression in x
        return abs(float(sp.N(d, 30))) < tol
    except (TypeError, ValueError): return False

# Each question: the question, the answers in order, the working, (for rounded answers) how close counts, and the move
# that cracks each part (shown when that part is wrong).
Q = {}

def practice(n, answer=""):
    """Question n: an answer box, a check button and a show-working button."""
    q = Q[n]
    box = Text(value=answer, placeholder="your answer", description="answer:", layout=Layout(width="460px"))
    check, work, out = Button(description="check"), Button(description="show working"), Output()
    def on_check(_=None):
        out.clear_output()
        with out:
            try: got = read(box.value)
            except Exception: print("I can't read that yet: check the brackets and the signs."); return
            want, tol = q["answer"], q.get("tol", [1e-9] * len(q["answer"]))
            if len(got) != len(want): print(f"✗ the question asks for {len(want)} answer(s); you typed {len(got)}."); return
            ok = [same(g, w, t) for g, w, t in zip(got, want, tol)]
            if all(ok): print("✓ right"); return
            for k, (good, move) in enumerate(zip(ok, q["moves"]), 1):
                print(f"✓ part {k}: right" if good else f"✗ part {k}: not quite. The move that cracks it: {move}")
    def on_work(_=None):
        out.clear_output()
        with out:
            for line in q["working"]: print(line)
    check.on_click(on_check); work.on_click(on_work)
    print(f"Question {n}. {q['text']}")
    display(VBox([HBox([box, check, work]), out]))
    if answer: on_check()

def expand(expr): return sp.expand(read_one(prep(expr)))

## 1 · Expanding (a + b)ⁿ by multiplying out

(a + b)² = a² + 2ab + b²: a square of side a + b cut into a square a², a square b² and two strips ab. Not a² + b²: the
strips are missing. Multiply by (a + b) again and (a + b)³ = a³ + 3a²b + 3ab² + b³, a cube cut into eight blocks.
Once more: (a + b)⁴ = a⁴ + 4a³b + 6a²b² + 4ab³ + b⁴. Three patterns: there are n + 1 terms; the power of a falls by
one each term while the power of b rises; and in every term the two powers add up to n. The numbers in front, 1 4 6 4
1, read the same backwards. Each new row of them comes from the row before: 1 3 3 1 shifted and added to itself.

The first cell builds those numbers into Pascal's triangle. Move the slider, and new rows appear, each number made by
adding the two above it. Beside it, the newest row is written out as the expansion of (a + b)ⁿ.

In [ ]:
def triangle(rows):
    """Pascal's triangle, built by adding: each number is the sum of the two above it."""
    t = [[1]]
    for _ in range(rows):
        prev = t[-1]
        t.append([1] + [prev[k - 1] + prev[k] for k in range(1, len(prev))] + [1])
    return t

T = triangle(12)
assert all(T[n][r] == comb(n, r) for n in range(13) for r in range(n + 1))          # the triangle is n choose r
assert T[4] == [1, 4, 6, 4, 1] and T[6] == [1, 6, 15, 20, 15, 6, 1]                 # the video's rows
assert all(sum(T[n]) == 2 ** n for n in range(13))
a_, b_ = sp.symbols("a b")
assert all(sp.Poly(sp.expand((a_ + b_) ** n), a_, b_).coeffs() == T[n] for n in range(9))   # the rows are the expansions

def term_tex(c, n, k):
    """c aⁿ⁻ᵏ bᵏ as mathtext: 1 and powers of one left out."""
    pa = "" if n - k == 0 else "a" if n - k == 1 else f"a^{{{n - k}}}"
    pb = "" if k == 0 else "b" if k == 1 else f"b^{{{k}}}"
    return ("" if c == 1 and (pa or pb) else str(c)) + pa + pb

def pascal(rows=4):
    t = triangle(rows)
    fig = plt.figure(figsize=(11.5, 5.2))
    ax = fig.add_axes([0.0, 0.09, 0.55, 0.86]); ax.axis("off")
    fs = 17 if rows <= 5 else 15 if rows <= 7 else 12.5 if rows <= 9 else 11
    H = max(rows, 5)                                                    # the triangle fills the space it has
    ax.set_xlim(-(H * 0.55 + 0.8), H * 0.55 + 0.8); ax.set_ylim(-H - 0.6, 0.7)
    pos = lambda m, k: ((k - m / 2) * 1.1, -m)
    for m in range(1, rows + 1):                                        # the joins: each number to the two above it
        for k in range(m + 1):
            for pk in (k - 1, k):
                if 0 <= pk <= m - 1:
                    new = m == rows
                    (x0, y0), (x1, y1) = pos(m - 1, pk), pos(m, k)
                    ax.plot([x0, x1], [y0 - 0.22, y1 + 0.25], color=GOLD if new else GHOST, lw=1.6 if new else 1.0,
                            alpha=0.75 if new else 1)
    for m, row in enumerate(t):
        for k, v in enumerate(row):
            ax.text(*pos(m, k), str(v), ha="center", va="center", fontsize=fs, color=GOLD if m == rows else CHALK,
                    fontweight="bold" if m == rows else "normal")
    if rows >= 2:
        sums = [f"{t[rows - 1][k - 1]} + {t[rows - 1][k]} = {t[rows][k]}" for k in range(1, rows)]
        fig.text(0.02, 0.03, f"new row {rows}:  1,  " + ",  ".join(sums[:3]) + (",  …" if len(sums) > 3 else ",  1"),
                 color=GOLD, fontsize=13)
    tp = fig.add_axes([0.57, 0.0, 0.43, 1.0]); tp.axis("off"); tp.set_xlim(0, 1); tp.set_ylim(0, 1)
    tp.text(0.0, 0.9, f"row {rows} is the numbers in front of", color=DIM, fontsize=14)
    tp.text(0.0, 0.79, f"$(a + b)^{{{rows}}}$  =", color=CHALK, fontsize=22)
    terms = [term_tex(c, rows, k) for k, c in enumerate(t[rows])]
    per = 3 if rows <= 7 else 2                                         # a few terms to a line
    for i in range(0, len(terms), per):
        line = " + ".join(terms[i:i + per])
        lead = "+ " if i else "  "
        tp.text(0.04, 0.66 - 0.105 * (i // per), f"{lead}${line}$", color=GOLD, fontsize=19 if rows <= 8 else 17)
    tp.text(0.0, 0.04, f"{rows + 1} term{'s' if rows else ''}; the numbers add up to 2{sup(rows)} = {2 ** rows}",
            color=DIM, fontsize=14)
    plt.show()

In [ ]:
interact(pascal, rows=IntSlider(value=4, min=0, max=10, description="rows"));

In [ ]:
Q[1] = {"text": "A square courtyard in Marrakesh is 52 metres on each side. Find its area by writing 52 = 50 + 2 and "
                "using (a + b)². Then a box is a cube with edges x + 4 centimetres long: expand its volume, (x + 4)³. "
                "And expand (2x − 3)³.",
        "answer": [(50 + 2) ** 2, expand("(x + 4)^3"), expand("(2x - 3)^3")],
        "moves": ["(a + b)² = a² + 2ab + b² with a = 50 and b = 2.",
                  "(a + b)³ = a³ + 3a²b + 3ab² + b³ with a = x and b = 4.",
                  "the same pattern with a = 2x and b = −3: the odd powers of −3 are negative."],
        "working": ["52² = (50 + 2)² = 50² + 2 × 50 × 2 + 2² = 2500 + 200 + 4 = 2704 square metres.",
                    "(x + 4)³ = x³ + 3 × x² × 4 + 3 × x × 4² + 4³ = x³ + 12x² + 48x + 64.",
                    "Check with x = 1: 5³ = 125, and 1 + 12 + 48 + 64 = 125.",
                    "(2x − 3)³ = (2x)³ + 3(2x)²(−3) + 3(2x)(−3)² + (−3)³ = 8x³ − 36x² + 54x − 27.",
                    "Check with x = 1: (−1)³ = −1, and 8 − 36 + 54 − 27 = −1."]}
assert Q[1]["answer"] == [2704, x ** 3 + 12 * x ** 2 + 48 * x + 64, 8 * x ** 3 - 36 * x ** 2 + 54 * x - 27]
assert same(read("x^3 + 12x^2 + 48x + 64")[0], Q[1]["answer"][1]) and not same(read("x^3 + 64")[0], Q[1]["answer"][1])
practice(1)

## 2 · Pascal's triangle

Stack the numbers from the expansions and you get Pascal's triangle: each number is the sum of the two above it,
because each term of (a + b)ⁿ⁺¹ can be reached two ways, from the row above times a or times b. Use a row to expand
any power: (x + 2)⁴ takes row 4, 1 4 6 4 1, with the powers of 2 rising: x⁴ + 8x³ + 24x² + 32x + 16. A minus sign
makes the signs alternate. Row n reads the same backwards, adds up to 2ⁿ, and counts coin flips: row 4's 6 is the
number of ways four coins can show two heads. Its weakness: to get row 12 you need every row above it.

In [ ]:
FLIPS = ["".join(f) for f in product("HT", repeat=5)]
Q[2] = {"text": "A bank in Toronto adds 2% to your savings every month. By what number are your savings multiplied "
                "after three months? Write 1.02 as 1 + 0.02 and use row 3 of the triangle. Then a coin is tossed 5 "
                "times: in how many orders can exactly 2 heads come up? And expand (x − 2)⁴.",
        "answer": [sp.Rational(102, 100) ** 3, sum(1 for f in FLIPS if f.count("H") == 2), expand("(x - 2)^4")],
        "moves": ["row 3 is 1 3 3 1: 1 + 3(0.02) + 3(0.02)² + (0.02)³.",
                  "read it from row 5: 1 5 10 10 5 1, place 2 (counting from 0).",
                  "row 4 is 1 4 6 4 1, with the powers of −2 rising: 1, −2, 4, −8, 16."],
        "working": ["(1 + 0.02)³ = 1 + 3(0.02) + 3(0.0004) + 0.000008 = 1 + 0.06 + 0.0012 + 0.000008 = 1.061208.",
                    "So the savings are multiplied by 1.061208: a little more than 6%, because of interest on interest.",
                    "Two heads in 5 tosses: row 5 is 1 5 10 10 5 1, and its place 2 is 10.",
                    "(Listing them: HHTTT, HTHTT, HTTHT, HTTTH, THHTT, THTHT, THTTH, TTHHT, TTHTH, TTTHH, 10.)",
                    "(x − 2)⁴ with row 4: x⁴ + 4x³(−2) + 6x²(4) + 4x(−8) + 16 = x⁴ − 8x³ + 24x² − 32x + 16.",
                    "Check with x = 1: (−1)⁴ = 1, and 1 − 8 + 24 − 32 + 16 = 1."]}
assert Q[2]["answer"] == [sp.Rational(1061208, 10 ** 6), 10, x ** 4 - 8 * x ** 3 + 24 * x ** 2 - 32 * x + 16]
assert T[3] == [1, 3, 3, 1] and T[5][2] == 10
practice(2)

## 3 · Counting: factorials and combinations

n things can be lined up in n! = n × (n − 1) × … × 1 ways, and 0! = 1. The number of ways to choose r things from n,
when order doesn't matter, is n choose r (some books write ⁿCᵣ): n! / (r! (n − r)!). Choosing r to take is choosing
n − r to leave, so n choose r = n choose n − r. And Pascal's rule, n choose r − 1 plus n choose r = (n + 1) choose r,
shows that the triangle's numbers are exactly the n choose r's: row n, place r.

In [ ]:
Q[3] = {"text": "In how many orders can 5 friends line up for a photo in Rio? Then: a stall in Manila needs 4 "
                "volunteers, chosen from 9 who offered. How many different teams are possible? And 11 choose 4 = 330, "
                "11 choose 5 = 462: use Pascal's rule to find 12 choose 5.",
        "answer": [factorial(5), comb(9, 4), comb(11, 4) + comb(11, 5)],
        "moves": ["5 choices for the first place, then 4, then 3 …: 5!.",
                  "order doesn't matter in a team: 9! ÷ (4! × 5!).",
                  "Pascal's rule: the two numbers side by side in row 11 add up to the one below them."],
        "working": ["Photo: 5 × 4 × 3 × 2 × 1 = 5! = 120 orders.",
                    "Team: 9 choose 4 = (9 × 8 × 7 × 6) ÷ (4 × 3 × 2 × 1) = 3024 ÷ 24 = 126.",
                    "Pascal's rule: 11 choose 4 + 11 choose 5 = 12 choose 5, so 330 + 462 = 792.",
                    "Check: 12 choose 5 = (12 × 11 × 10 × 9 × 8) ÷ 120 = 95040 ÷ 120 = 792."]}
assert Q[3]["answer"] == [120, 126, 792] == [120, 3024 // 24, comb(12, 5)]
practice(3)

## 4 · The binomial theorem

(a + b)ⁿ = aⁿ + (n choose 1)aⁿ⁻¹b + (n choose 2)aⁿ⁻²b² + … + bⁿ, the sum from r = 0 to n of (n choose r) aⁿ⁻ʳ bʳ. Why:
multiplying out n brackets means choosing a or b from each one, and the number of ways to get aⁿ⁻ʳbʳ is the number
of ways to choose which r brackets give b. A minus sign belongs to b, so its odd powers are negative. Put a = b = 1:
the numbers in row n add up to 2ⁿ. And to add up the numbers in front of the powers of x in any expansion, put x = 1.

In [ ]:
Q[4] = {"text": "Expand (1 − 3x)⁴. Then a phone battery in Berlin keeps 97% of its capacity each year: after four "
                "years it keeps 0.97⁴ of it. Use your expansion with x = 0.01 to find 0.97⁴ to four decimal places. "
                "And without expanding, add up the numbers in front of the powers of x in (3x − 1)⁵.",
        "answer": [expand("(1 - 3x)^4"), sp.Rational(97, 100) ** 4, expand("(3x - 1)^5").subs(x, 1)],
        "tol": [1e-9, 0.00006, 1e-9],
        "moves": ["the numbers 1 4 6 4 1, with the powers of −3x: 1, −3x, 9x², −27x³, 81x⁴.",
                  "1 − 12(0.01) + 54(0.01)² − 108(0.01)³ + 81(0.01)⁴, then round.",
                  "put x = 1: (3 − 1)⁵."],
        "working": ["(1 − 3x)⁴ = 1 + 4(−3x) + 6(9x²) + 4(−27x³) + 81x⁴ = 1 − 12x + 54x² − 108x³ + 81x⁴.",
                    "With x = 0.01: 1 − 0.12 + 0.0054 − 0.000108 + 0.00000081 = 0.88529281.",
                    "To four decimal places, 0.8853: the battery keeps about 88.5% of its capacity.",
                    "Put x = 1 in (3x − 1)⁵: every term becomes just its number, so they add up to (3 − 1)⁵ = 2⁵ = 32."]}
assert Q[4]["answer"] == [1 - 12 * x + 54 * x ** 2 - 108 * x ** 3 + 81 * x ** 4, sp.Rational(88529281, 10 ** 8), 32]
assert round(0.97 ** 4, 4) == 0.8853
practice(4)

## 5 · General and middle terms

The general term is Tᵣ₊₁ = (n choose r) aⁿ⁻ʳ bʳ. It's term number r + 1, so the fourth term has r = 3. For a
particular power of x, write the general term's power of x in terms of r and solve for r. The term independent of x
is the one whose power of x is zero; r must be a whole number from 0 to n, so sometimes there isn't one. There are n
+ 1 terms: when n is even, one middle term, number n/2 + 1; when n is odd, two, numbers (n + 1)/2 and (n + 3)/2.

The cell below writes out the general term of any bracket. Choose n with its slider and type the two terms of the
bracket (a number times a power of x, like `x^2`, `-3/x`, `2x`), pressing Enter after each. It writes the general
term with its power of x. Move the slider for r, and it shows each term in turn.

In [ ]:
r_ = sp.Symbol("r")

def monomial(e):
    """A term c·x^k → (c, k), or None if it isn't a number times a power of x."""
    c, k = sp.sympify(e).as_coeff_exponent(x)
    if c.has(x) or not k.is_number or sp.simplify(c * x ** k - e) != 0: return None
    return sp.nsimplify(c), sp.nsimplify(k)

def term_list(ca, ka, cb, kb, n):
    """Every term of (ca x^ka + cb x^kb)ⁿ: (r, number in front, power of x)."""
    return [(r, comb(n, r) * ca ** (n - r) * cb ** r, ka * (n - r) + kb * r) for r in range(n + 1)]

assert [(c, p) for _, c, p in term_list(1, 2, 2, -1, 6) if p == 0] == [(240, 0)]          # the video: (x² + 2/x)⁶
assert [(c, p) for _, c, p in term_list(1, 1, 1, -1, 6) if p == 0] == [(20, 0)]           # (x + 1/x)⁶
assert not [p for _, _, p in term_list(1, 1, 1, -1, 5) if p == 0]                         # (x + 1/x)⁵: none
assert [c for _, c, p in term_list(1, 1, -2, -2, 9) if p == 0] == [-672]                  # (x − 2/x²)⁹

def paren(c, power):
    """A number raised to a power, in brackets when it's negative or a fraction."""
    s = sp.latex(c)
    return (f"\\left({s}\\right)" if (c < 0 or not c.is_integer) else s) + f"^{{{power}}}"

def br(e):
    """A bracket's term as mathtext, in brackets unless it's just x."""
    return "x" if e == x else f"\\left({sp.latex(e)}\\right)"

def general(n=5, first="x", second="1/x", r=0):
    try:
        A, B = read_one(prep(first)), read_one(prep(second)); ma, mb = monomial(A), monomial(B)
    except Exception:
        ma = mb = None
    if ma is None or mb is None:
        print("Type each term of the bracket as a number times a power of x, like x^2, -3/x or 2x, then press Enter.")
        return
    (ca, ka), (cb, kb), r = ma, mb, min(r, n)
    terms = term_list(ca, ka, cb, kb, n)
    bl = sp.latex(B); bracket = sp.latex(A) + (" " + bl if bl.startswith("-") else " + " + bl)
    P = sp.expand(ka * (n - r_) + kb * r_)
    fig = plt.figure(figsize=(11.5, 5.4))
    tp = fig.add_axes([0, 0, 1, 1]); tp.axis("off"); tp.set_xlim(0, 1); tp.set_ylim(0, 1)
    tp.text(0.02, 0.9, f"$\\left({bracket}\\right)^{{{n}}}$", color=CHALK, fontsize=24)
    gen = "" if ca == 1 else paren(ca, f"{n}-r") + r"\,"
    gen += "" if cb == 1 else paren(cb, "r") + r"\,"
    xp = "" if P == 0 else "x" if P == 1 else f"x^{{{sp.latex(P)}}}"
    tp.text(0.27, 0.9, f"$T_{{r+1}} = \\binom{{{n}}}{{r}}\\,{br(A)}^{{{n}-r}}{br(B)}^{{r}} = \\binom{{{n}}}{{r}}\\,{gen}{xp}$",
            color=SKY, fontsize=21)
    tp.text(0.02, 0.77, "the power of x in each term:", color=DIM, fontsize=14)
    w = 0.96 / (n + 1)
    for k, c, p in terms:                                               # one chip for each term, r = 0 … n
        cx = 0.02 + w * (k + 0.5)
        now, zero = k == r, p == 0
        if now: tp.add_patch(plt.Rectangle((cx - w * 0.46, 0.555), w * 0.92, 0.18, fc=SLATE2, ec=CORAL, lw=2.4))
        tp.text(cx, 0.69, f"$T_{{{k + 1}}}$", ha="center", color=CORAL if now else DIM, fontsize=14)
        tp.text(cx, 0.6, f"$x^{{{sp.latex(p)}}}$", ha="center", color=GOLD if zero else CORAL if now else CHALK,
                fontsize=17 if n <= 8 else 14, fontweight="bold")
    k, c, p = terms[r]
    tp.text(0.02, 0.42, f"$r = {r}:\\quad T_{{{r + 1}}} = \\binom{{{n}}}{{{r}}}\\,{br(A)}^{{{n - r}}}{br(B)}^{{{r}}}"
                        f" = {sp.latex(c * x ** p)}$", color=CORAL, fontsize=22)
    if p == 0: tp.text(0.02, 0.32, "this term is independent of x: its power of x is zero", color=GOLD, fontsize=16)
    msg2 = ""
    if ka == kb: msg = "every term has the same power of x"
    else:
        r0 = sp.Rational(ka * n, ka - kb)                               # the power of x is zero at r = r0
        msg = f"independent of x:  ${sp.latex(P)} = 0$  when  $r = {sp.latex(r0)}$"
        if r0.is_integer and 0 <= r0 <= n: msg += f",  so  $T_{{{r0 + 1}}} = {sp.latex(terms[int(r0)][1])}$"
        else: msg2 = f"but r must be a whole number from 0 to {n}, so no term is independent of x"
    tp.text(0.02, 0.2, msg, color=DIM if msg2 else GOLD, fontsize=17)
    if msg2: tp.text(0.02, 0.12, msg2, color=DIM, fontsize=15)
    mid = f"the middle term is $T_{{{n // 2 + 1}}}$" if n % 2 == 0 else f"the middle terms are $T_{{{(n + 1) // 2}}}$ and $T_{{{(n + 3) // 2}}}$"
    tp.text(0.02, 0.04, f"{n + 1} terms; {mid}", color=DIM, fontsize=14)
    plt.show()

NG = IntSlider(value=5, min=1, max=12, description="n")
RG = IntSlider(value=0, min=0, max=5, description="r")
NG.observe(lambda ch: setattr(RG, "max", ch["new"]), names="value")     # r runs from 0 to n

In [ ]:
interact(general, n=NG, first=Text(value="x", description="first term", continuous_update=False),
         second=Text(value="1/x", description="second term", continuous_update=False), r=RG);

In [ ]:
Q5_TERMS = term_list(1, 1, -3, -2, 9)
Q[5] = {"text": "Find the term independent of x in the expansion of (x − 3/x²)⁹. Then find the middle term of (2x + 1)⁶.",
        "answer": [[c for _, c, p in Q5_TERMS if p == 0][0], comb(6, 3) * 2 ** 3 * x ** 3],
        "moves": ["the general term is (9 choose r) x⁹⁻ʳ (−3/x²)ʳ: its power of x is 9 − 3r. Make it zero.",
                  "7 terms, so the middle one is T₄, with r = 3: (6 choose 3)(2x)³."],
        "working": ["Tᵣ₊₁ = (9 choose r) x⁹⁻ʳ (−3)ʳ x⁻²ʳ = (9 choose r)(−3)ʳ x⁹⁻³ʳ.",
                    "Independent of x: 9 − 3r = 0, so r = 3. That's T₄.",
                    "T₄ = (9 choose 3)(−3)³ = 84 × (−27) = −2268.",
                    "(2x + 1)⁶ has 7 terms, so the middle one is T₄, with r = 3.",
                    "The power of 2x is 6 − 3 = 3: T₄ = (6 choose 3)(2x)³(1)³ = 20 × 8x³ = 160x³."]}
assert Q[5]["answer"] == [-2268, 160 * x ** 3] and comb(9, 3) * (-3) ** 3 == -2268
practice(5)

## 6 · Using the binomial theorem

When x is small, (1 + x)ⁿ ≈ 1 + nx, and each later term is much smaller than the one before, so a few terms give a
good approximation: 1.01¹⁰ ≈ 1 + 0.1 + 0.0045 + 0.00012 = 1.10462. Big powers come apart the same way: 98⁵ =
(100 − 2)⁵. The first two terms settle which of two huge numbers is larger: 1.01¹⁰⁰⁰⁰⁰⁰ > 1 + 1000000 × 0.01 = 10001.
And for divisibility, write the base as 1 plus a multiple: 6ⁿ = (1 + 5)ⁿ = 1 + 5n + (terms with 25 in them).

In [ ]:
SEOUL = [comb(8, r) * sp.Rational(3, 100) ** r for r in range(9)]
Q[6] = {"text": "Savings in Seoul earn 3% a year. By what number are they multiplied after 8 years? Give 1.03⁸ to "
                "three decimal places, using the first few terms. Then work out 99⁴ by writing 99 = 100 − 1. And "
                "what's the remainder when 8¹⁰ is divided by 49? (Write 8 = 1 + 7.)",
        "answer": [sum(SEOUL), (100 - 1) ** 4, 8 ** 10 % 49], "tol": [0.0006, 1e-9, 1e-9],
        "moves": ["1 + 8(0.03) + 28(0.03)² + 56(0.03)³ + …: the terms after that are too small to change the third decimal.",
                  "(100 − 1)⁴ with row 4: 100⁴ − 4 × 100³ + 6 × 100² − 4 × 100 + 1.",
                  "(1 + 7)¹⁰ = 1 + 10 × 7 + (terms with 7² = 49 in them). So the remainder is what's left of 1 + 70."],
        "working": ["1.03⁸ = 1 + 8(0.03) + 28(0.0009) + 56(0.000027) + 70(0.00000081) + …",
                    "= 1 + 0.24 + 0.0252 + 0.001512 + 0.0000567 + … = 1.26677…, so 1.267 to three decimal places.",
                    "99⁴ = 100000000 − 4000000 + 60000 − 400 + 1 = 96059601.",
                    "8¹⁰ = (1 + 7)¹⁰ = 1 + 70 + (10 choose 2)7² + … and every term from the third on contains 49.",
                    "So 8¹⁰ leaves the same remainder as 71 does, and 71 = 49 + 22: the remainder is 22."]}
assert round(float(sum(SEOUL)), 3) == 1.267 == round(1.03 ** 8, 3)
assert Q[6]["answer"][1:] == [96059601, 22] and 99 ** 4 == 96059601 and (1 + 70) % 49 == 22
assert all(comb(10, r) * 7 ** r % 49 == 0 for r in range(2, 11))
practice(6)

## Sources
NCERT Mathematics Class XI, ch. 7 (Binomial Theorem); OpenStax, Precalculus 2e, 11.6 (Binomial Theorem). Both
retrieved 2026-11-01. Every question is our own.